# 🟡 Interagindo com Neo4j usando Python

Este notebook é um guia didático e prático que demonstra como interagir com o **Neo4j** (banco de dados NoSQL do tipo **Grafos**) utilizando a linguagem Python, o driver oficial `neo4j` e a linguagem de consulta **Cypher**.

## 🛠️ O que é o Neo4j?
O Neo4j armazena dados estruturados como **redes de conexões**. Em vez de tabelas (SQL) ou documentos (MongoDB), os dados são modelados como:
- **Nós (Nodes):** Entidades (ex: Pessoa, Empresa, Produto).
- **Relacionamentos (Relationships):** Conexões direcionadas com tipos específicos entre nós (ex: `AMIGO_DE`, `TRABALHA_NA`, `COMPROU`).
- **Propriedades (Properties):** Pares chave-valor associados tanto a nós quanto a relacionamentos.

### Resumo Conceitual

| Propriedade | Detalhes |
|---|---|
| **Paradigma** | Grafo (Graph Database) |
| **Linguagem de Consulta** | Cypher (declarativa, visual, baseada em padrões ASCII) |
| **Armazenamento** | Nós e relacionamentos como cidadãos de primeira classe (não como tabelas de junção) |
| **Quando usar** | Redes sociais, recomendações, detecção de fraudes, grafos de conhecimento, análise de dependências, rotas e logística |
| **Quando NÃO usar** | Dados tabulares simples, operações de agregação massiva (OLAP), dados sem relações significativas |

### Vocabulário Visual do Cypher

```
(n)              → Nó
(n:Pessoa)       → Nó com Label (tipo) 'Pessoa'
(n {nome: 'X'})  → Nó com propriedade
-[r]->           → Relacionamento direcionado
-[r:AMIGO_DE]->  → Relacionamento com tipo 'AMIGO_DE'
```

### Detalhes da Conexão Local (Docker Compose):
- **Host:** `localhost`
- **Porta Bolt:** `7688` (protocolo binário de alta performance usado pelo driver)
- **Porta HTTP:** `7475` (painel web do Neo4j Browser)
- **Autenticação:** Nenhuma (`NEO4J_AUTH=none` no docker-compose.yml)

## 📋 Pré-requisitos

Antes de executar este notebook, certifique-se de que:

1. O **Docker** está instalado e em execução na sua máquina.
2. Os containers do projeto foram iniciados com `make up` ou `docker compose up -d`.
3. O container `neo4j` está rodando (verifique com `docker compose ps`).

> **💡 Dica:** Você pode visualizar os grafos criados neste notebook de forma interativa acessando [http://localhost:7475](http://localhost:7475) no navegador.

## 2. Conectando ao Banco de Dados
Vamos importar a classe `GraphDatabase` e criar uma instância de driver apontando para a porta **Bolt** (protocolo binário otimizado do Neo4j).

> **💡 Conceito-Chave:** O protocolo **Bolt** (porta 7687) é o canal de comunicação de alta performance entre o driver Python e o Neo4j. A porta HTTP (7475) é usada apenas pelo painel web do navegador.

> **Nota:** Como desativamos a autenticação no docker-compose (`NEO4J_AUTH=none`), passamos `auth=None`.

**Saída esperada:**
```
✅ Conexão com Neo4j estabelecida com sucesso!
⚡ Versão do Neo4j: 5.26.x
```

In [ ]:
import time
from neo4j import GraphDatabase

# URI de conexão usando protocolo Bolt (binário, alta performance)
uri = "bolt://localhost:7688"

max_retries = 3
for attempt in range(1, max_retries + 1):
    try:
        # Criar o driver de conexão (sem autenticação no container local)
        driver = GraphDatabase.driver(uri, auth=None, connection_timeout=10, max_connection_lifetime=60)
        
        # Verificar conexão executando uma query rápida
        with driver.session() as session:
            resultado = session.run(
                "CALL dbms.components() YIELD name, versions RETURN versions[0] AS versao"
            ).single()
            print("✅ Conexão com Neo4j estabelecida com sucesso!")
            print(f"⚡ Versão do Neo4j: {resultado['versao']}")
            break
    except Exception as e:
        print(f"⚠️ Tentativa {attempt}/{max_retries} falhou. O Neo4j pode estar inicializando...")
        if attempt == max_retries:
            print(f"❌ Erro final ao conectar ao Neo4j: {e}")
            print("Certifique-se de que o container do Neo4j está rodando e as portas estão mapeadas corretamente.")
        else:
            time.sleep(5)  # Aguardar 5 segundos antes de tentar novamente


---
## 3. Limpando o Banco de Dados de Testes
Antes de criar novos nós e arestas, vamos rodar um comando Cypher para limpar todos os dados existentes, garantindo que o notebook execute de forma limpa e reproduzível.

> **💡 Conceito-Chave: Transações.** No Neo4j, toda operação ocorre dentro de uma **transação**. O método `execute_write()` garante que a operação será executada como uma transação de escrita — se houver erro, as alterações serão revertidas (rollback).

**Saída esperada:**
```
🧹 Banco de dados de grafos limpo e pronto!
```

In [ ]:
def limpar_banco(tx):
    """Remove todos os nós e relacionamentos do banco de dados.
    DETACH DELETE remove primeiro os relacionamentos e depois os nós."""
    tx.run("MATCH (n) DETACH DELETE n")

# execute_write() executa a função dentro de uma transação de escrita
# O parâmetro 'tx' (transaction) é injetado automaticamente pelo driver
with driver.session() as session:
    session.execute_write(limpar_banco)
    print("🧹 Banco de dados de grafos limpo e pronto!")

---
## 4. CRUD — Create (Criar Nós e Relacionamentos)
Utilizaremos queries Cypher **parametrizadas** (com `$nome_variavel`). A parametrização é recomendada para evitar ataques de injeção e melhorar o cache de planos de execução do Neo4j.

Vamos construir a seguinte rede social:

```
    ┌─────────┐    AMIGO_DE     ┌─────────┐    AMIGO_DE    ┌─────────┐
    │  Alice  │ ──────────────► │   Bob   │ ─────────────► │ Charlie │
    └─────────┘                 └─────────┘                └─────────┘
         │
         │ AMIGO_DE
         ▼
    ┌─────────┐
    │  Diego  │
    └─────────┘
```

**Saída esperada:**
```
👤 Nós de pessoas criados com sucesso!
🔗 Relacionamentos de amizade criados com sucesso!
```

In [ ]:
def criar_pessoa(tx, nome, idade, cidade):
    """Cria um nó com label 'Pessoa' e as propriedades especificadas.
    CREATE sempre cria um novo nó (use MERGE para evitar duplicatas)."""
    query = """
    CREATE (p:Pessoa {nome: $nome, idade: $idade, cidade: $cidade})
    RETURN p
    """
    # Parâmetros com $ são substituídos de forma segura pelo driver
    tx.run(query, nome=nome, idade=idade, cidade=cidade)

def criar_amizade(tx, nome1, nome2, desde_ano):
    """Cria um relacionamento AMIGO_DE entre duas pessoas existentes.
    O relacionamento é DIRECIONADO (a -> b) e possui a propriedade 'desde'."""
    query = """
    MATCH (a:Pessoa {nome: $nome1})
    MATCH (b:Pessoa {nome: $nome2})
    CREATE (a)-[r:AMIGO_DE {desde: $desde_ano}]->(b)
    RETURN r
    """
    tx.run(query, nome1=nome1, nome2=nome2, desde_ano=desde_ano)

# Executar as operações de criação dentro de transações de escrita
with driver.session() as session:
    # 1. Criar os Nós de Pessoa
    session.execute_write(criar_pessoa, "Alice", 28, "João Pessoa")
    session.execute_write(criar_pessoa, "Bob", 30, "Recife")
    session.execute_write(criar_pessoa, "Charlie", 25, "Natal")
    session.execute_write(criar_pessoa, "Diego", 35, "João Pessoa")
    print("👤 Nós de pessoas criados com sucesso!")
    
    # 2. Criar os Relacionamentos de Amizade (Arestas direcionadas)
    session.execute_write(criar_amizade, "Alice", "Bob", 2022)
    session.execute_write(criar_amizade, "Bob", "Charlie", 2023)
    session.execute_write(criar_amizade, "Alice", "Diego", 2021)
    print("🔗 Relacionamentos de amizade criados com sucesso!")

---
## 5. CRUD — Read (Consultar Dados e Caminhos)
Vamos realizar duas consultas que demonstram o verdadeiro poder dos bancos de grafos:

1. **Buscar amigos diretos** da Alice (travessia de 1 nível).
2. **Recomendação de amigos** — buscar "amigos de amigos" com quem Alice ainda não é conectada (travessia de 2 níveis).

> **💡 Conceito-Chave:** É exatamente nesse tipo de consulta (travessia de grafos) que o Neo4j se destaca em relação a bancos relacionais. Em SQL, uma recomendação de "amigo de amigo" exigiria múltiplos JOINs — no Cypher, é um padrão visual simples.

**Saída esperada:**
```
📖 Amigos diretos da Alice:
- Bob (30 anos), amigos desde 2022
- Diego (35 anos), amigos desde 2021
--------------------------------------------------
📊 Recomendações de amizade para Alice:
- Sugerido: Charlie (Porque é amigo de Bob)
```

In [ ]:
# === Consulta 1: Amigos diretos da Alice ===
# O padrão (p)-[r:AMIGO_DE]->(amigo) lê-se:
# "encontre pessoa p que tem relação AMIGO_DE apontando para amigo"
query_amigos = """
MATCH (p:Pessoa {nome: $nome_alvo})-[r:AMIGO_DE]->(amigo)
RETURN amigo.nome AS nome, amigo.idade AS idade, r.desde AS ano_amizade
"""

with driver.session() as session:
    resultados = session.run(query_amigos, nome_alvo="Alice")
    print("📖 Amigos diretos da Alice:")
    for registro in resultados:
        print(f"- {registro['nome']} ({registro['idade']} anos), amigos desde {registro['ano_amizade']}")

print("-" * 50)

# === Consulta 2: Recomendação (Amigos de Amigos da Alice) ===
# O padrão com dois saltos: Alice -> Amigo -> AmigoDoAmigo
# O WHERE garante que:
#   1. Não estamos recomendando alguém com quem Alice já é amiga
#   2. Não estamos recomendando a própria Alice
query_recomendacao = """
MATCH (alice:Pessoa {nome: 'Alice'})-[:AMIGO_DE]->(amigo)-[:AMIGO_DE]->(sugerido)
WHERE NOT (alice)-[:AMIGO_DE]->(sugerido) AND sugerido.nome <> 'Alice'
RETURN sugerido.nome AS recomendacao, amigo.nome AS intermediario
"""

with driver.session() as session:
    recomendacoes = session.run(query_recomendacao)
    print("📊 Recomendações de amizade para Alice:")
    for rec in recomendacoes:
        print(f"- Sugerido: {rec['recomendacao']} (Porque é amigo de {rec['intermediario']})")

---
## 6. CRUD — Update (Atualizar Nós ou Relacionamentos)
Atualizações no Neo4j utilizam a cláusula `SET` aplicada a nós ou relacionamentos encontrados via `MATCH`.

> **💡 Conceito-Chave:** `MATCH` encontra os nós/relações existentes, e `SET` modifica suas propriedades. Você pode atualizar tanto **nós** quanto **relacionamentos**.

**Saída esperada:**
```
🔄 Idade de Bob atualizada com sucesso para 31 anos!
```

In [ ]:
# === Atualizar a idade do Bob para 31 anos ===
# MATCH encontra o nó, SET modifica a propriedade
query_update_idade = """
MATCH (p:Pessoa {nome: $nome})
SET p.idade = $nova_idade
RETURN p.nome AS nome, p.idade AS idade
"""

with driver.session() as session:
    resultado = session.run(query_update_idade, nome="Bob", nova_idade=31).single()
    print(f"🔄 Idade de {resultado['nome']} atualizada com sucesso para {resultado['idade']} anos!")

---
## 7. CRUD — Delete (Deletar Nós e Relacionamentos)
No Neo4j, você **não pode deletar um nó** se ele ainda estiver conectado por relacionamentos (isso garante a **integridade referencial** do grafo — não podem existir arestas "soltas" apontando para o nada).

Para contornar isso, usamos `DETACH DELETE`, que:
1. Primeiro exclui **todas** as conexões do nó.
2. Depois deleta o nó em si.

**Saída esperada:**
```
🗑️ Nó de Charlie e todas as suas conexões de amizade foram removidos!

👥 Pessoas restantes no banco de dados:
- Alice
- Bob
- Diego
```

In [ ]:
# === Deletar o nó do Charlie ===
# DETACH DELETE remove o nó E todos os seus relacionamentos
# Sem DETACH, o Neo4j lançaria um erro se o nó tiver conexões
query_delete = """
MATCH (p:Pessoa {nome: $nome_deletar})
DETACH DELETE p
"""

with driver.session() as session:
    session.run(query_delete, nome_deletar="Charlie")
    print("🗑️ Nó de Charlie e todas as suas conexões de amizade foram removidos!")

    # Mostrar pessoas restantes no grafo
    print("\n👥 Pessoas restantes no banco de dados:")
    todos = session.run("MATCH (p:Pessoa) RETURN p.nome AS nome")
    for pessoa in todos:
        print(f"- {pessoa['nome']}")

---
## 8. Encerrando a Conexão
É uma boa prática fechar o driver ao final do uso para liberar conexões com o servidor Neo4j.

In [ ]:
# Fechar o driver Neo4j (encerra todas as sessões e conexões)
driver.close()
print("🔌 Conexão com Neo4j encerrada com sucesso.")

---
## 🏁 Conclusão
Parabéns! Você concluiu os testes com o Neo4j e a linguagem Cypher:
- ✅ Aprendeu a instanciar e gerenciar conexões via protocolo binário **Bolt**.
- ✅ Criou **nós** estruturados de forma dinâmica com Labels e propriedades.
- ✅ Criou **relacionamentos** tipados e direcionados entre entidades no grafo.
- ✅ Entendeu como navegar por conexões usando padrões em Cypher, incluindo consultas para **recomendação** (amigo de amigo).
- ✅ Atualizou atributos de nós e realizou remoções seguras usando `DETACH DELETE`.

### 🚀 Próximos Passos
Para continuar se aprofundando no Neo4j, experimente:
1. **Shortest Path:** Encontrar o caminho mais curto entre dois nós com `shortestPath()`.
2. **MERGE:** Criar nós/relações apenas se não existirem (evitando duplicatas).
3. **Constraints e Índices:** Garantir unicidade e performance com `CREATE CONSTRAINT`.
4. **APOC Library:** Extensão com centenas de procedures utilitárias para análise de grafos.
5. **Graph Data Science:** Algoritmos como PageRank, Community Detection e Centrality.

### 📚 Referências Úteis
- [Documentação oficial do Neo4j](https://neo4j.com/docs/)
- [Guia de Cypher](https://neo4j.com/docs/cypher-manual/current/)
- [Neo4j Python Driver](https://neo4j.com/docs/python-manual/current/)
- [Neo4j Browser (Interface Web)](http://localhost:7475)
- [Neo4j Sandbox (playground gratuito)](https://neo4j.com/sandbox/)